# LLooM (library): Inciting Speech — per-classifier FN/FP analysis

Uses **`text_lloom`** on **classifier reasoning only** (`model_response_identity`, `model_response_imputed_misdeeds`, `model_response_exhortation`) for **false negatives and false positives** across the three inciting-speech binary classifiers.

**Data:** `combined2.jsonl` in the same directory — each record has `gold_label`, `pred_identity`, `pred_imputed_misdeeds`, `pred_exhortation`, and the three reasoning fields.

**Models:** `gpt-3.5-turbo` (distill + score), `gpt-5-mini` (synthesize), `text-embedding-3-large` (cluster). `text_lloom` does not ship defaults for `gpt-5-mini`; the pipeline patches tiktoken to `o200k_base` and supplies explicit `context_window / cost / rate_limit`.

**Outputs:** `outputs/revised_lloom/{classifier}/{fn_fp}/` — `lloom_export_summary.csv`, `lloom_distilled_bullets.csv`, `lloom_scores_long.csv`, and a pickled session `lloom_session.pkl`.

Set **`OPENAI_API_KEY`** before running (see next cell).

In [15]:
import os

# Recommended: set in your shell before launching Jupyter:
#   export OPENAI_API_KEY="sk-..."
#
# Or set for this Python session only (never commit a real key):



In [16]:
!pip install -q -U text_lloom nest_asyncio ipywidgets pandas tiktoken



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [17]:
import asyncio
import json
import os
import re
import types

import nest_asyncio
import pandas as pd

nest_asyncio.apply()

import text_lloom.workbench as wb
import text_lloom.concept_induction as _ci
from text_lloom.llm import OpenAIModel, OpenAIEmbedModel
from text_lloom.prompts import filter_prompt as default_filter_prompt
from text_lloom.workbench import lloom as _Lloom


## Configuration

In [18]:
# ── Data & output ──────────────────────────────────────────────────────────
JSONL_PATH   = "combined2.jsonl"
OUTPUT_BASE  = "outputs/revised_lloom"

# None = run full file; set e.g. 50 for a cheap test run
LIMIT_JSONL  = None

# ── Models ─────────────────────────────────────────────────────────────────
MODEL_DISTILL = "gpt-3.5-turbo"
MODEL_SYNTH   = "gpt-5-mini"
MODEL_SCORE   = "gpt-3.5-turbo"
MODEL_EMBED   = "text-embedding-3-large"

# ── Per-classifier seeds (short — text_lloom uppercases and wraps them) ────
SEED_MAP = {
    "identity": "missed identity group cues, implicit dehumanisation, ambiguous group reference",
    "imputed_misdeeds": "missed accusation of harmful acts, subtle blame attribution, indirect imputation",
    "exhortation": "missed call to action, implicit urgency, indirect incitement",
}

# ── Max highlights fed to summarize_concept (avoids token-limit errors) ────
SUMMARIZE_MAX_EXAMPLES = 80


## Load JSONL and extract per-classifier FN/FP subsets

In [19]:
# ── load_jsonl: robust parser for concatenated JSON objects ──────────────
def load_jsonl(path):
    """Handles both strict JSONL and concatenated `{...}\n{...}` files."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    decoder = json.JSONDecoder()
    idx = 0
    while idx < len(content):
        while idx < len(content) and content[idx] in " \t\n\r":
            idx += 1
        if idx >= len(content):
            break
        obj, end_idx = decoder.raw_decode(content, idx)
        records.append(obj)
        idx = end_idx
    return records


def _clean_text_for_openai(s):
    """OpenAI returns 400 invalid_request_error if content is NaN or bad UTF-8."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    if not isinstance(s, str):
        s = str(s)
    return s.encode("utf-8", errors="replace").decode("utf-8")


# ── Load ─────────────────────────────────────────────────────────────────
records = load_jsonl(JSONL_PATH)
if LIMIT_JSONL is not None:
    records = records[:LIMIT_JSONL]
    print(f"Truncated to first {len(records)} record(s) (LIMIT_JSONL={LIMIT_JSONL})")

for i, d in enumerate(records):
    d["orig_index"] = i

print(f"Loaded {len(records)} total records.")


Loaded 7000 total records.


In [20]:
# ── Per-classifier FN/FP split ────────────────────────────────────────────
FIELD_MAP = {
    "identity":        "model_response_identity",
    "imputed_misdeeds": "model_response_imputed_misdeeds",
    "exhortation":     "model_response_exhortation",
}
GOLD_LABEL_MAP = {
    "identity":        "Identity",
    "imputed_misdeeds": "Imputed Misdeeds",
    "exhortation":     "Exhortation",
}
# Positive prediction string for each binary classifier
PRED_POSITIVE_MAP = {
    "identity":        "Identity",
    "imputed_misdeeds": "Imputed Misdeeds",
    "exhortation":     "Exhortation",
}

datasets = []
for clf in ["identity", "imputed_misdeeds", "exhortation"]:
    gold_label   = GOLD_LABEL_MAP[clf]
    pred_pos     = PRED_POSITIVE_MAP[clf]
    resp_field   = FIELD_MAP[clf]
    pred_field   = f"pred_{clf}"

    # FN: gold IS target, model missed it
    fn_rows = [
        {"doc_id": str(d["orig_index"]), "text": _clean_text_for_openai(d[resp_field]),
         "gold_label": d["gold_label"], pred_field: d.get(pred_field, "")}
        for d in records
        if d.get("gold_label") == gold_label
        and d.get(pred_field) != pred_pos
        and d.get(resp_field)
    ]

    # FP: gold NOT target, model predicted positive
    fp_rows = [
        {"doc_id": str(d["orig_index"]), "text": _clean_text_for_openai(d[resp_field]),
         "gold_label": d["gold_label"], pred_field: d.get(pred_field, "")}
        for d in records
        if d.get("gold_label") != gold_label
        and d.get(pred_field) == pred_pos
        and d.get(resp_field)
    ]

    for mode, rows in [("fn", fn_rows), ("fp", fp_rows)]:
        df_sub = pd.DataFrame(rows)
        if df_sub.empty:
            print(f"  Warning: no {mode.upper()} examples for {clf}, skipping.")
            continue
        df_sub = df_sub[df_sub["text"].str.strip() != ""]
        datasets.append({
            "name":       f"{clf}_{mode}",
            "classifier": clf,
            "fn_fp":      mode,
            "df":         df_sub,
            "seed":       SEED_MAP[clf],
        })

print(f"Created {len(datasets)} subsets:")
for ds in datasets:
    print(f"  {ds['name']}: {len(ds['df'])} examples")


Created 6 subsets:
  identity_fn: 162 examples
  identity_fp: 2884 examples
  imputed_misdeeds_fn: 467 examples
  imputed_misdeeds_fp: 1872 examples
  exhortation_fn: 141 examples
  exhortation_fp: 1177 examples


## Custom prompts

Required placeholders: `{ex}`, `{n_bullets}`, `{seeding_phrase}`, `{n_words}` (distill); `{examples}`, `{n_concepts_phrase}`, `{seeding_phrase}` (synthesize).

`text_lloom` passes `seeding_phrase` as `("related to " + seed).upper()` for distill, and *The patterns MUST BE RELATED TO {SEED_UPPERCASE}* for synthesize.

In [21]:
INCITING_DISTILL_PROMPT = """
Task: Error-analysis distillation. The text below is ONLY the classifier's written
rationale for a false-negative or false-positive inciting-speech decision.

Summarize that rationale into {n_bullets} bullet phrases. Describe **claims, logic,
and cues** the model used — not the original post, not your own judgment.

Rules:
- Each bullet must be about {n_words} words
- Capture how the model justified its label (e.g., tone, absent key phrase, ambiguity)
- Avoid copying long quotes; paraphrase
- Avoid generic bullets ("model analyzed text"); be specific
- Do not invent content not supported by the rationale
- Return valid JSON only

**Thematic constraint (from the induction seed):**
{seeding_phrase}

Return format:
{{
  "bullets": ["...", "..."]
}}

CLASSIFIER_REASONING:
{ex}
"""

INCITING_SYNTH_PROMPT = """
You cluster error-analysis bullets from **inciting-speech classifier** runs.
Each bullet summarizes the model's rationale for a false-negative or false-positive decision.

Propose {n_concepts_phrase} describing **recurring justification strategies** in the
model's reasoning (not themes of raw posts).

**Thematic constraint (from the induction seed):**
{seeding_phrase}

Ground each pattern in the bullet summaries in the JSON below; do not invent mechanisms
absent from the cluster.

For each pattern object in the output array, use keys exactly: **name**, **prompt**, **example_ids**.
- **name**: 2-6 words, specific (not "miscellaneous errors")
- **prompt**: ONE sentence, yes/no question about the rationale text (used for automated scoring).
  Example: "Does this reasoning dismiss an identity reference as non-specific?"
- **example_ids**: 1-2 strings copied exactly from `example_id` fields in the input JSON below

**CRITICAL — output format:**
- Your entire reply must be a single JSON object. First character MUST be `{{` and last MUST be `}}`.
- No markdown, no numbered lists, no 'Name:'/'Prompt:' sections, no code fences — only raw JSON.

Input examples (JSON):
{examples}

Return exactly this structure:
{{
 "patterns": [
   {{"name": "...", "prompt": "...", "example_ids": ["...", "..."]}}
 ]
}}
"""

custom_prompts = {
    "distill_filter":    default_filter_prompt,
    "distill_summarize": INCITING_DISTILL_PROMPT,
    "synthesize":        INCITING_SYNTH_PROMPT,
}


## LLooM helper functions (gpt-5-mini patch, cost skip, summarize_concept cap)

In [23]:
def _patch_text_lloom_for_gpt5_synth(model_name: str) -> None:
    """text_lloom's defaults + tiktoken do not list gpt-5-*; patch tokenizers to o200k_base."""
    if not str(model_name).startswith("gpt-5"):
        return
    import tiktoken
    import text_lloom.llm_openai as lo
    import text_lloom.llm as tlm

    enc = tiktoken.get_encoding("o200k_base")

    def count_tokens_fn(model, text):
        return len(enc.encode(text))

    def truncate_tokens_fn(model, text, out_token_alloc=1500):
        tokens = enc.encode(text)
        max_t = model.context_window - out_token_alloc
        if len(tokens) > max_t:
            tokens = tokens[:max_t]
        return enc.decode(tokens)

    lo.count_tokens_fn = count_tokens_fn
    lo.truncate_tokens_fn = truncate_tokens_fn
    tlm.count_tokens_fn = count_tokens_fn
    tlm.truncate_tokens_fn = truncate_tokens_fn


def make_synth_openai_model(name: str, api_key: str) -> OpenAIModel:
    """OpenAIModel() for gpt-5-* with explicit context/cost/rate and no-temperature call."""
    _patch_text_lloom_for_gpt5_synth(name)

    async def call_llm_fn_synth(model, prompt):
        if "system_prompt" not in model.args:
            model.args["system_prompt"] = (
                "You help identify patterns in text. When the user asks for structured output, "
                "reply with a single valid JSON object only: no markdown, no numbered lists, "
                "no 'Name:'/'Prompt:' sections, and no code fences — only raw JSON."
            )
        prompt = model.truncate_fn(model, prompt, out_token_alloc=1500)
        res = await model.client.chat.completions.create(
            model=model.name,
            messages=[
                {"role": "system", "content": model.args["system_prompt"]},
                {"role": "user",   "content": prompt},
            ],
        )
        res_parsed  = res.choices[0].message.content if res else None
        in_tokens   = res.usage.prompt_tokens if res else 0
        out_tokens  = res.usage.completion_tokens if res else 0
        return res_parsed, (in_tokens, out_tokens)

    if str(name).startswith("gpt-5"):
        _1m = 1_000_000
        return OpenAIModel(
            name=name,
            api_key=api_key,
            fn=call_llm_fn_synth,
            context_window=128_000,
            cost=(0.25 / _1m, 2.0 / _1m),
            rate_limit=(20, 10),
        )
    return OpenAIModel(name=name, api_key=api_key)


# ── Skip cost-estimate prompts (avoids blocking user input) ──────────────
def _lloom_skip_estimate_gen_cost(self, params=None, verbose=False):
    return None


def _lloom_skip_estimate_score_cost(
    self, n_concepts=None, batch_size=5, get_highlights=True, verbose=False, df_to_score=None
):
    return None


# ── Cap highlights fed to summarize_concept (same as HateXplain notebook) ─
if getattr(_ci, "_lloom_orig_summarize_concept", None) is None:
    _ci._lloom_orig_summarize_concept = _ci.summarize_concept


async def _summarize_concept_subsampled(
    score_df,
    concept_id,
    model,
    sess=None,
    threshold=1.0,
    summary_length="15-20 word",
    score_col="score",
    highlight_col="highlight",
):
    df = score_df.copy()
    df = df[df[score_col] >= threshold]
    cur_df = df[df["concept_id"] == concept_id]
    if len(cur_df) > SUMMARIZE_MAX_EXAMPLES:
        keep_ix = cur_df.sample(n=SUMMARIZE_MAX_EXAMPLES, random_state=42).index
        drop_ix = cur_df.index.difference(keep_ix)
        score_df = score_df.drop(index=drop_ix)
    return await _ci._lloom_orig_summarize_concept(
        score_df, concept_id, model, sess=sess,
        threshold=threshold, summary_length=summary_length,
        score_col=score_col, highlight_col=highlight_col,
    )


_ci.summarize_concept = _summarize_concept_subsampled
print("Helper functions defined.")


Helper functions defined.


## LLooM pipeline — run all 6 subsets sequentially

- **`filter_n_quotes: 1`** — reasoning texts are already single documents; default auto-suggest would try to split them into quotes, which is wrong here.
- **`summ_n_bullets: 3`** — must be an integer for cost-estimation arithmetic.
- **`auto_review: True`** — LLooM removes/merges low-quality concepts automatically.
- Score is run with `batch_size=1` and `get_highlights=True` for reliable per-doc scores.

In [ ]:
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY before running (e.g. export OPENAI_API_KEY=sk-...)")

all_instances = {}


async def run_pipeline(ds):
    clf, mode, df, seed = ds["classifier"], ds["fn_fp"], ds["df"], ds["seed"]
    out_dir = f"{OUTPUT_BASE}/{clf}/{mode}"
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"🚀  {clf.upper()} | {mode.upper()}  —  {len(df)} reasoning texts")

    l = wb.lloom(
        df=df,
        text_col="text",
        id_col="doc_id",
        distill_model=OpenAIModel(name=MODEL_DISTILL, api_key=api_key),
        cluster_model=OpenAIEmbedModel(name=MODEL_EMBED, api_key=api_key),
        synth_model=make_synth_openai_model(MODEL_SYNTH, api_key),
        score_model=OpenAIModel(name=MODEL_SCORE, api_key=api_key),
    )

    # Patch out blocking cost-estimate prompts
    l.estimate_gen_cost   = types.MethodType(_lloom_skip_estimate_gen_cost,   l)
    l.estimate_score_cost = types.MethodType(_lloom_skip_estimate_score_cost, l)

    suggested = l.auto_suggest_parameters()
    params = {
        "filter_n_quotes": 1,
        "summ_n_bullets":  3,
        "synth_n_concepts": min(max(suggested["synth_n_concepts"], 2), 8),
    }
    print("  LLooM params:", params)

    # ── Gen ────────────────────────────────────────────────────────────────
    await l.gen(
        seed=seed,
        params=params,
        n_synth=1,
        custom_prompts=custom_prompts,
        auto_review=True,
        debug=False,
    )

    if not l.concepts:
        print("  ⚠️  No concepts after gen (check synthesis errors); skipping score.")
        return l

    # Activate all concepts for scoring
    l.select_widget = None
    for c in l.concepts.values():
        c.active = True

    # ── Score ──────────────────────────────────────────────────────────────
    await l.score(score_all=True, debug=False, get_highlights=True, batch_size=1)

    # ── Export ─────────────────────────────────────────────────────────────
    export_df = l.export_df()
    export_df.to_csv(f"{out_dir}/lloom_export_summary.csv", index=False)

    if getattr(l, "df_bullets", None) is not None:
        l.df_bullets.to_csv(f"{out_dir}/lloom_distilled_bullets.csv", index=False)
    if getattr(l, "results", None):
        l.get_score_df().to_csv(f"{out_dir}/lloom_scores_long.csv", index=False)

    # ── Save session (restore real estimate methods before pickling) ────────
    _egc, _esc = l.estimate_gen_cost, l.estimate_score_cost
    l.estimate_gen_cost   = _Lloom.estimate_gen_cost.__get__(l,   _Lloom)
    l.estimate_score_cost = _Lloom.estimate_score_cost.__get__(l, _Lloom)
    try:
        l.save(folder=out_dir, file_name="lloom_session")
    finally:
        l.estimate_gen_cost, l.estimate_score_cost = _egc, _esc

    l.summary(verbose=True)
    print(f"  ✅  Saved to {out_dir}/")
    return l


loop = asyncio.get_event_loop()
for ds in datasets:
    inst = loop.run_until_complete(run_pipeline(ds))
    all_instances[ds["name"]] = inst

print("\n✅  All iterations complete.")



🚀  IDENTITY | FN  —  162 reasoning texts
  LLooM params: {'filter_n_quotes': 1, 'summ_n_bullets': 3, 'synth_n_concepts': 6}


Distill-summarize
✅ Done    


Cluster
✅ Done    


Synthesize
✅ Done    
✅ Done with concept generation!
100%|██████████| 18/18 [02:56<00:00,  9.81s/it]
✅ Done with concept scoring!
Saved session to outputs/revised_lloom/identity/fn/lloom_session.pkl
Total time: 220.31 sec (3.67 min)
	('Distill-summarize', '2026-04-02-22-22-28'): 2.91 sec
	('Cluster', '2026-04-02-22-22-33'): 4.97 sec
	('Synthesize', '2026-04-02-22-23-08'): 35.77 sec
	('Score', '2026-04-02-22-26-05'): 176.65 sec


Total cost: $1.85
	('Distill-summarize', '2026-04-02-22-22-28'): $0.068
	('Synthesize', '2026-04-02-22-23-08'): $0.013
	('Score-helper', '2026-04-02-22-23-11'): $0.098
	('Score-helper', '2026-04-02-22-23-21'): $0.098
	('Score-helper', '2026-04-02-22-23-32'): $0.101
	('Score-helper', '2026-04-02-22-23-41'): $0.098
	('Score-helper', '2026-04-02-22-23-51'): $0.101
	('Score-helper', '2026

## Interactive visualization

Change `TARGET` to any of: `'identity_fn'`, `'identity_fp'`, `'imputed_misdeeds_fn'`, `'imputed_misdeeds_fp'`, `'exhortation_fn'`, `'exhortation_fp'`.

In [ ]:
TARGET = "identity_fn"  # ← change this

inst = all_instances.get(TARGET)
if inst:
    inst.vis()
else:
    print(f"Instance '{TARGET}' not found — check that the pipeline completed successfully.")
